# Molecular Dynamics: 3D Lennard-Jones Fluid

A reproducible **NVE (constant particle number, volume, and total energy)** simulation using NumPy, periodic boundary conditions, a face-centered cubic (FCC) lattice, and velocity Verlet integration.

**Units:** Lennard-Jones reduced units with $m=\sigma=\epsilon=k_B=1$. The initial temperature controls the sampled velocities; there is **no thermostat**, so temperature can fluctuate during evolution.

For quick execution, use $4(3^3)=108$ particles. To reproduce the original 864-particle system, set `n_cells = 6` (expect slower execution because the direct pair-force calculation scales as $O(N^2)$).

## 1. Physics

The Lennard-Jones potential is

$$U(r)=4\epsilon\left[\left(\frac{\sigma}{r}\right)^{12}-\left(\frac{\sigma}{r}\right)^6\right].$$

The force on particle $i$ from $j$, with $\mathbf r_{ij}=\mathbf r_i-\mathbf r_j$, is

$$\mathbf F_{ij}=24\epsilon\left[2\left(\frac{\sigma}{r_{ij}}\right)^{12}-\left(\frac{\sigma}{r_{ij}}\right)^6\right]\frac{\mathbf r_{ij}}{r_{ij}^2}.$$

The **minimum-image convention** is used for periodic boundaries. A cutoff $r_c=2.5\sigma$ is used, and the potential is shifted so $U(r_c)=0$. The force is not force-shifted, so small cutoff-related energy drift can remain.

Velocity Verlet updates are

$$\mathbf r_{t+\Delta t}=\mathbf r_t+\mathbf v_t\Delta t+\frac{\mathbf F_t}{2m}\Delta t^2,$$

$$\mathbf v_{t+\Delta t}=\mathbf v_t+\frac{\mathbf F_t+\mathbf F_{t+\Delta t}}{2m}\Delta t.$$


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Simulation parameters (Lennard-Jones reduced units)
n_cells = 3             # 3 -> 108 particles; 6 -> 864 particles
number_density = 0.85
initial_temperature = 1.0
dt = 0.003
n_steps = 250
cutoff = 2.5
mass = 1.0
seed = 42

rng = np.random.default_rng(seed)
n_particles = 4 * n_cells**3
box_length = (n_particles / number_density)**(1 / 3)
assert cutoff < box_length / 2, "Cutoff must be smaller than half the box length."
print(f"Particles: {n_particles}; box length: {box_length:.4f}; cutoff: {cutoff}")

## 2. FCC lattice and initial velocities

In [ ]:
def initialize_fcc(n_cells, box_length):
    """Construct 4*n_cells**3 FCC sites inside a periodic cubic box."""
    a = box_length / n_cells
    basis = np.array([[0, 0, 0], [0, 0.5, 0.5],
                      [0.5, 0, 0.5], [0.5, 0.5, 0]])
    sites = []
    for i in range(n_cells):
        for j in range(n_cells):
            for k in range(n_cells):
                sites.extend((np.array([i, j, k]) + basis) * a)
    return np.asarray(sites, dtype=float)


def initialize_velocities(n_particles, temperature, rng, mass=1.0):
    """Sample Gaussian velocities, remove COM motion, rescale to target kinetic temperature."""
    velocities = rng.normal(0, np.sqrt(temperature / mass), (n_particles, 3))
    velocities -= velocities.mean(axis=0)
    kinetic = 0.5 * mass * np.sum(velocities**2)
    degrees_of_freedom = 3 * n_particles - 3
    current_temperature = 2 * kinetic / degrees_of_freedom
    velocities *= np.sqrt(temperature / current_temperature)
    return velocities

positions = initialize_fcc(n_cells, box_length)
velocities = initialize_velocities(n_particles, initial_temperature, rng, mass)
assert positions.shape == (n_particles, 3)
assert np.all((positions >= 0) & (positions < box_length))
assert np.allclose(velocities.mean(axis=0), 0, atol=1e-12)
print("FCC positions and velocities initialized.")

## 3. Periodic pair forces and potential energy

In [ ]:
def forces_and_potential(positions, box_length, cutoff=2.5):
    """Return net LJ forces and shifted LJ potential using unique i<j pairs."""
    n = len(positions)
    i, j = np.triu_indices(n, k=1)
    displacement = positions[i] - positions[j]
    displacement -= box_length * np.rint(displacement / box_length)
    r2 = np.einsum("ij,ij->i", displacement, displacement)
    if np.any(r2 < 1e-12):
        raise ValueError("Overlapping particles detected; check initialization.")
    mask = r2 < cutoff**2
    i, j, displacement, r2 = i[mask], j[mask], displacement[mask], r2[mask]
    inv_r2 = 1.0 / r2
    inv_r6 = inv_r2**3
    inv_r12 = inv_r6**2
    pair_force = (24 * (2 * inv_r12 - inv_r6) * inv_r2)[:, None] * displacement
    forces = np.zeros_like(positions)
    np.add.at(forces, i, pair_force)
    np.add.at(forces, j, -pair_force)
    inv_rc6 = cutoff**-6
    u_cutoff = 4 * (inv_rc6**2 - inv_rc6)
    potential = np.sum(4 * (inv_r12 - inv_r6) - u_cutoff)
    return forces, float(potential)


def kinetic_energy(velocities, mass=1.0):
    return 0.5 * mass * np.sum(velocities**2)

forces, potential = forces_and_potential(positions, box_length, cutoff)
assert np.allclose(forces.sum(axis=0), 0, atol=1e-9), "Net internal force should vanish."
print(f"Initial potential energy: {potential:.4f}")
print(f"Initial kinetic energy: {kinetic_energy(velocities, mass):.4f}")

## 4. Velocity Verlet simulation

In [ ]:
def verlet_step(positions, velocities, forces, box_length, dt, mass, cutoff):
    new_positions = (positions + velocities * dt + 0.5 * forces * dt**2 / mass) % box_length
    new_forces, new_potential = forces_and_potential(new_positions, box_length, cutoff)
    new_velocities = velocities + 0.5 * (forces + new_forces) * dt / mass
    return new_positions, new_velocities, new_forces, new_potential

# Store energies every step; save a modest number of configurations for visualization.
times = np.arange(n_steps + 1) * dt
kinetic_history = np.empty(n_steps + 1)
potential_history = np.empty(n_steps + 1)
temperature_history = np.empty(n_steps + 1)
frames = [positions.copy()]
frame_times = [0.0]
frame_stride = max(1, n_steps // 40)

kinetic_history[0] = kinetic_energy(velocities, mass)
potential_history[0] = potential
temperature_history[0] = 2 * kinetic_history[0] / (3 * n_particles - 3)

for step in range(1, n_steps + 1):
    positions, velocities, forces, potential = verlet_step(
        positions, velocities, forces, box_length, dt, mass, cutoff
    )
    kinetic_history[step] = kinetic_energy(velocities, mass)
    potential_history[step] = potential
    temperature_history[step] = 2 * kinetic_history[step] / (3 * n_particles - 3)
    if step % frame_stride == 0 or step == n_steps:
        frames.append(positions.copy())
        frame_times.append(times[step])

total_history = kinetic_history + potential_history
relative_drift = (total_history.max() - total_history.min()) / max(abs(total_history[0]), 1.0)
print(f"Total energy at start: {total_history[0]:.5f}")
print(f"Total energy at end:   {total_history[-1]:.5f}")
print(f"Relative total-energy range: {relative_drift:.3%}")
assert np.all(np.isfinite(total_history))
assert np.all((positions >= 0) & (positions < box_length))
assert np.allclose(forces.sum(axis=0), 0, atol=1e-7)
assert relative_drift < 0.05, "Large energy drift: try a smaller dt."
print("Basic physical and numerical checks passed.")

## 5. Energy conservation and temperature

In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(times, kinetic_history, label="Kinetic energy")
plt.plot(times, potential_history, label="Potential energy")
plt.plot(times, total_history, label="Total energy", linewidth=2)
plt.xlabel("Time (reduced units)")
plt.ylabel("Energy (reduced units)")
plt.title("Lennard-Jones molecular dynamics: energy evolution")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

plt.figure(figsize=(9, 4))
plt.plot(times, temperature_history)
plt.xlabel("Time (reduced units)")
plt.ylabel("Instantaneous kinetic temperature")
plt.title("Temperature fluctuations in an NVE simulation")
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

## 6. 3D visualization of particle configurations

Positions are wrapped into the simulation box. Do not connect wrapped coordinates with lines: crossing periodic boundaries would produce artificial long trajectories.

In [ ]:
fig = plt.figure(figsize=(11, 5))
for panel, (frame, label) in enumerate([(frames[0], "Initial FCC configuration"),
                                         (frames[-1], "Final configuration")], start=1):
    ax = fig.add_subplot(1, 2, panel, projection="3d")
    ax.scatter(frame[:, 0], frame[:, 1], frame[:, 2], s=12, alpha=0.7)
    ax.set(xlim=(0, box_length), ylim=(0, box_length), zlim=(0, box_length),
           xlabel="x", ylabel="y", zlabel="z", title=label)
    ax.set_box_aspect((1, 1, 1))
plt.tight_layout()
plt.show()

## 7. Interpretation and limitations

- **FCC initialization** prevents the severe particle overlaps of random high-density placement.
- **Energy conservation** is a useful correctness check for the NVE integrator. Some fluctuations/drift can arise from the finite time step and the truncated, potential-shifted force.
- **Temperature is not held constant**; the simulation is microcanonical after initial velocity preparation.
- This is a **teaching-scale direct $O(N^2)$ implementation**, not an optimized production MD engine. For larger systems, use neighbor lists and more sophisticated cutoff handling.
- The simulation is short and not guaranteed to reach equilibrium. Production measurements require equilibration and longer sampling.

**Run in VS Code:** select a Python/Jupyter kernel and run all cells. Install requirements with `python -m pip install numpy matplotlib jupyter`.
